<a href="https://colab.research.google.com/github/CMDDclass/MS697-material/blob/main/Hands-on-session5/Hands-on-session5-BO_practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice : Hyperparameter Optimization with BO (VAE)

We'll use a GPU for faster model training.

To enable the GPU:

1. Navigate to Runtime.

2. Select Change runtime type.

3. Choose T4 GPU from the dropdown menu.

4. Click Save.


## **Submission
- Sensitivity analysis
- Reconstructed image with the lowest validation loss  
- Latent space distribution corresponding to the lowest validation loss

In [ ]:
!pip install ax-platform

In [ ]:
# 0. Check GPU status
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
!nvidia-smi

In [ ]:
# 1. Library Imports and Environment Setup
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.losses import mse, binary_crossentropy
from tensorflow.keras.utils import plot_model
from scipy.stats import norm

from ax.api.client import Client
from ax.api.configs import ChoiceParameterConfig, RangeParameterConfig

In [ ]:
# Load and Preprocess MNIST Dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
image_size = x_train.shape[1]
original_dim = image_size * image_size
x_train = np.reshape(x_train, [-1, original_dim]).astype('float32') / 255
x_test = np.reshape(x_test, [-1, original_dim]).astype('float32') / 255
input_shape = (original_dim,)
# Reparameterization Trick
def sampling(args):
    """
    Reparameterization trick by sampling from an isotropic unit Gaussian.
    # Arguments:
        args (tensor): mean and log of variance of Q(z|X)
    # Returns:
        z (tensor): sampled latent vector
    """
    z_mean, z_log_var = args
    batch = keras.ops.shape(z_mean)[0]
    dim = keras.ops.shape(z_mean)[1]
    # by default, random_normal has mean=0 and std=1.0
    epsilon = keras.random.normal(shape=(batch, dim))
    return z_mean + keras.ops.exp(0.5 * z_log_var) * epsilon

# Encoder Function Definition
def build_encoder(input_shape, intermediate_dim, latent_dim):
    """
    Builds and returns the VAE encoder model.
    (Input) -> Dense -> (z_mean, z_log_var) -> z
    """
    inputs = Input(shape=input_shape, name='encoder_input')
    x = Dense(intermediate_dim, activation='relu')(inputs)
    z_mean = Dense(latent_dim, name='z_mean')(x)
    z_log_var = Dense(latent_dim, name='z_log_var')(x)
    z = Lambda(sampling, output_shape=(latent_dim,), name='z')([z_mean, z_log_var])

    # Create Keras Model object
    encoder = Model(inputs, [z_mean, z_log_var, z], name='encoder')
    return encoder

# Decoder Function Definition
def build_decoder(intermediate_dim, original_dim, latent_dim):
    """
    Builds and returns the VAE decoder model.
    (Latent Input) -> Dense -> (Output Image)
    """
    latent_inputs = Input(shape=(latent_dim,), name='z_sampling')
    x = Dense(intermediate_dim, activation='relu')(latent_inputs)
    outputs = Dense(original_dim, activation='sigmoid')(x)

    # Create Keras Model object
    decoder = Model(latent_inputs, outputs, name='decoder')
    return decoder

  # VAE Model Class Definition (Same as before)
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")

    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker,
        ]

    def call(self, data):
        """Defines the forward pass for inference."""
        _, _, z = self.encoder(data)
        reconstruction = self.decoder(z)
        return reconstruction

    def _compute_loss(self, data):
        z_mean, z_log_var, z = self.encoder(data)
        reconstruction = self.decoder(z)
        reconstruction_loss = keras.ops.sum(
            keras.ops.binary_crossentropy(data, reconstruction), axis=-1
        )
        kl_loss = -0.5 * (1 + z_log_var - keras.ops.square(z_mean) - keras.ops.exp(z_log_var))
        kl_loss = keras.ops.sum(kl_loss, axis=-1)
        total_loss = keras.ops.mean(reconstruction_loss + kl_loss)
        return total_loss, keras.ops.mean(reconstruction_loss), keras.ops.mean(kl_loss)

    def train_step(self, data):
        with tf.GradientTape() as tape:
            total_loss, reconstruction_loss, kl_loss = self._compute_loss(data)
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

    def test_step(self, data):
        total_loss, reconstruction_loss, kl_loss = self._compute_loss(data)
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
        }

def plot_label_clusters(vae, data, labels):
    """
    Plots the distribution of test data in the 2D latent space.
    """
    z_mean, _, _ = vae.encoder.predict(data, verbose=0)
    plt.figure(figsize=(12, 10))
    plt.scatter(z_mean[:, 0], z_mean[:, 1], c=labels, cmap='tab10', alpha=0.7)
    plt.colorbar(ticks=range(10))
    plt.xlabel("Latent Dimension 1")
    plt.ylabel("Latent Dimension 2")
    plt.title("Latent Space Distribution of MNIST Digits")
    plt.grid(True)
    plt.show()


In [ ]:
def run_vae_experiment(
    latent_dim=2,
    batch_size=128,
    intermediate_dim=512,
    epochs=10,
    visualize=True,
):
    print("---" * 30)
    print(f"Starting Experiment with: latent_dim={latent_dim}, intermediate_dim={intermediate_dim}, epochs={epochs}")
    print("---" * 30)


    encoder = build_encoder(input_shape, intermediate_dim, latent_dim)
    decoder = build_decoder(intermediate_dim, original_dim, latent_dim)
    vae = VAE(encoder, decoder)

    # 2. Compile and Train
    vae.compile(optimizer='adam')
    history = vae.fit(
        x_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(x_test, None),
        verbose=1,
    )

    print("Training Finished.")

    # 3. Quantitative Analysis
    val_loss = history.history['val_loss'][-1]
    print(f"  - Final Validation Loss: {val_loss:.4f}")

    if visualize:
        # 4. Qualitative Analysis: Visualize results
        print("  - Visualizing results...")

        # Reconstruction quality
        x_decoded = vae.predict(x_test, batch_size=batch_size, verbose=0)
        n = 10
        plt.figure(figsize=(20, 4))
        plt.suptitle(
            f"Reconstruction (latent_dim={latent_dim}, intermediate_dim={intermediate_dim})",
            fontsize=16,
        )
        for i in range(n):
            ax = plt.subplot(2, n, i + 1)
            plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
            ax.set_title("Original")
            ax.axis('off')
            ax = plt.subplot(2, n, i + 1 + n)
            plt.imshow(x_decoded[i].reshape(28, 28), cmap='gray')
            ax.set_title("Reconstructed")
            ax.axis('off')
        plt.show()

        plot_label_clusters(vae, x_test, y_test)

    return val_loss, history, vae



In [ ]:
client = Client()

In [ ]:
def vae_objective(latent_dim, batch_size, intermediate_dim, epochs):
    val_loss, _, _ = run_vae_experiment(
        latent_dim=int(latent_dim),
        batch_size=int(batch_size),
        intermediate_dim=int(intermediate_dim),
        epochs=int(epochs),
        visualize=False,
    )
    return val_loss

# You can change here-----------------------------------------------------------------------
client.configure_experiment(
    parameters=[

    ]
)
# ------------------------------------------------------------------------------------------

In [ ]:
client.configure_optimization(objective="-val_loss")

In [ ]:
# You can change here-----------------------------------------------------------------------
number_of_experiments =
# ------------------------------------------------------------------------------------------


for _ in range(number_of_experiments):
    trials = client.get_next_trials(max_trials=1)

    for trial_index, params in trials.items():
        print(f"\n=== BO Trial {trial_index} / params = {params} ===")

        val_loss, history, vae = run_vae_experiment(
            latent_dim=int(params["latent_dim"]),
            batch_size=int(params["batch_size"]),
            intermediate_dim=int(params["intermediate_dim"]),
            epochs=int(params["epochs"]),
            visualize=True,
        )

        client.complete_trial(
            trial_index=trial_index,
            raw_data={"val_loss": val_loss},
        )

        print(f"Trial {trial_index} completed with val_loss={val_loss:.4f}")

In [ ]:
# Obtain the best-performing configuration; the true minimum for the booth
best_parameters, prediction, index, name = client.get_best_parameterization()
print("Best Parameters:", best_parameters)
print("Prediction (mean, variance):", prediction)

In [ ]:
# display=True instructs Ax to sort then render the resulting analyses
cards = client.compute_analyses(display=True)